In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers -q

import torch
import torch.nn as nn
import json
import numpy as np
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

DRIVE_BASE = "/content/drive/MyDrive/medrag"
DATA_PATH = f"{DRIVE_BASE}/pubmedqa_filtered.json"
CHECKPOINT_DIR = f"{DRIVE_BASE}/tuned_lens_checkpoints"
OUTPUT_DIR = f"{DRIVE_BASE}/trial3_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(DATA_PATH, "r") as f:
    corpus = json.load(f)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Corpus loaded: {len(corpus)} samples")

In [ ]:
MODEL_ID = "stanford-crfm/BioMedLM"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    attn_implementation="eager"
)
model = model.to("cuda")
model.eval()

for param in model.parameters():
    param.requires_grad = False

n_layers = model.config.n_layer
hidden_size = model.config.n_embd
vocab_size = model.config.vocab_size

print(f"Model loaded: {n_layers} layers, hidden size {hidden_size}, vocab {vocab_size}")

In [ ]:
class TunedLensTranslator(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.translator = nn.Linear(hidden_size, hidden_size, bias=True)
        nn.init.eye_(self.translator.weight)
        nn.init.zeros_(self.translator.bias)

    def forward(self, hidden_state):
        return self.translator(hidden_state.float())

translators = nn.ModuleList([
    TunedLensTranslator(hidden_size) for _ in range(n_layers)
])

for layer_idx in range(n_layers):
    ckpt = torch.load(
        os.path.join(CHECKPOINT_DIR, f"translator_layer_{layer_idx:02d}.pt"),
        map_location="cuda"
    )
    translators[layer_idx].load_state_dict(ckpt["state_dict"])

translators = translators.to("cuda")
translators.eval()
for param in translators.parameters():
    param.requires_grad = False

LN_F_WEIGHT = model.transformer.ln_f.weight.float().detach()
LN_F_BIAS = model.transformer.ln_f.bias.float().detach()
LM_HEAD_WEIGHT = model.lm_head.weight.float().detach()

print(f"Tuned-lens loaded: {n_layers} translators ready")

In [ ]:
def project_hidden_with_tuned_lens(hidden_state, layer_idx):
    with torch.no_grad():
        translated = translators[layer_idx](hidden_state.float())
        mean = translated.mean(-1, keepdim=True)
        std = translated.std(-1, keepdim=True)
        normalized = (translated - mean) / (std + 1e-5)
        normed = normalized * LN_F_WEIGHT + LN_F_BIAS
        logits = normed @ LM_HEAD_WEIGHT.T
        probs = torch.softmax(logits, dim=-1)
    return probs

def kl_from_uniform(probs):
    """
    KL divergence between model distribution and uniform baseline.
    KL(p || uniform) = sum(p * log(p * vocab_size))
    High KL = peaked distribution, model has strong prediction.
    Low KL = flat distribution, model is uncertain (close to uniform).
    """
    log_uniform = torch.log(torch.tensor(1.0 / vocab_size, device=probs.device))
    kl = torch.sum(probs * (torch.log(probs + 1e-10) - log_uniform), dim=-1)
    return kl

print("Projection and KL functions ready.")
print(f"Uniform baseline: 1/{vocab_size} = {1/vocab_size:.8f} per token")

In [ ]:
hook_storage = {"hidden_states": {}}
hooks = []

def make_hook(layer_idx):
    def hook(module, input, output):
        hook_storage["hidden_states"][layer_idx] = output[0].detach()
    return hook

for i, block in enumerate(model.transformer.h):
    hooks.append(block.register_forward_hook(make_hook(i)))

N_SAMPLES = len(corpus)
trial3_results = []

print(f"Running Trial 3 on {N_SAMPLES} samples...\n")

for idx, sample in enumerate(tqdm(corpus[:N_SAMPLES])):
    query = sample["query"]

    prompt = f"Question: {query}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt",
                      truncation=True, max_length=512).to("cuda")
    seq_len = inputs["input_ids"].shape[1]

    hook_storage["hidden_states"].clear()
    with torch.no_grad():
        outputs = model(**inputs)

    kl_trajectory = []

    for layer_idx in range(n_layers):
        hidden = hook_storage["hidden_states"][layer_idx][0]
        probs = project_hidden_with_tuned_lens(hidden, layer_idx)

        last_token_probs = probs[-1].unsqueeze(0)
        kl = kl_from_uniform(last_token_probs)
        kl_trajectory.append(float(kl[0]))

    trial3_results.append({
        "idx": idx,
        "pubid": sample["pubid"],
        "query": query[:60],
        "label": sample["label"],
        "seq_len": seq_len,
        "kl_trajectory": kl_trajectory,
        "mean_kl": float(np.mean(kl_trajectory[1:])),
        "max_kl_layer": int(np.argmax(kl_trajectory[1:]) + 1),
        "max_kl_value": float(np.max(kl_trajectory[1:])),
        "final_layer_kl": kl_trajectory[-1]
    })

    if (idx + 1) % 50 == 0:
        ckpt_path = os.path.join(OUTPUT_DIR, f"trial3_checkpoint_{idx+1}.json")
        with open(ckpt_path, "w") as f:
            json.dump(trial3_results, f)
        torch.cuda.empty_cache()
        tqdm.write(f"Checkpoint saved at sample {idx+1}")

print(f"\nTrial 3 complete. {len(trial3_results)} samples processed.")

In [ ]:
print("TRIAL 3 SUMMARY BY LABEL \n")
for label in ["yes", "no"]:
    label_samples = [r for r in trial3_results if r["label"] == label]
    print(f"Label: {label} | n={len(label_samples)}")
    print(f"  Mean KL from uniform:  {np.mean([r['mean_kl'] for r in label_samples]):.4f}")
    print(f"  Mean peak layer:       {np.mean([r['max_kl_layer'] for r in label_samples]):.1f}")
    print(f"  Mean final layer KL:   {np.mean([r['final_layer_kl'] for r in label_samples]):.4f}")
    print()

output_path = os.path.join(OUTPUT_DIR, "trial3_full_results.json")
with open(output_path, "w") as f:
    json.dump({
        "n_samples": len(trial3_results),
        "n_layers": n_layers,
        "vocab_size": vocab_size,
        "summary_by_label": {
            label: {
                "n": sum(1 for r in trial3_results if r["label"] == label),
                "mean_kl": float(np.mean([r["mean_kl"] for r in trial3_results
                                         if r["label"] == label])),
                "mean_peak_layer": float(np.mean([r["max_kl_layer"] for r in trial3_results
                                                  if r["label"] == label])),
                "mean_final_kl": float(np.mean([r["final_layer_kl"] for r in trial3_results
                                                if r["label"] == label]))
            } for label in ["yes", "no"]
        },
        "samples": trial3_results
    }, f, indent=2)

for fname in os.listdir(OUTPUT_DIR):
    if fname.startswith("trial3_checkpoint"):
        os.remove(os.path.join(OUTPUT_DIR, fname))

for h in hooks:
    h.remove()

print(f"Results saved to {output_path}")
print("Trial 3 complete")

In [ ]:
from google.colab import runtime
runtime.unassign()